# Plan-and-Execute

Port of the LangGraph [Plan-and-Execute](https://langchain-ai.github.io/langgraph/tutorials/plan-and-execute/plan-and-execute/) agent pattern to LangGraph4j.

Related to [#8](https://github.com/langgraph4j/langgraph4j/issues/8).

## Agentic Architecture

```
START → planner → agent → replan ──► (continue) → agent
                      ↑               │
                      └───────────────┘
                                      └──► (respond) → END
```

| Node | Role |
|------|------|
| `planner` | Break the user goal into an ordered list of steps |
| `agent` | Execute the **first** remaining step (tool call / stub) |
| `replan` | Drop completed work, refresh the plan, or produce the final answer |

This notebook ships with a **deterministic stub** so the graph compiles and runs without an API key.
Replace the stub nodes with LangChain4j `AiServices` / `ChatModel` when you are ready for a live LLM.


In [ ]:
var userHomeDir = System.getProperty("user.home");
var localRespoUrl = "file://" + userHomeDir + "/.m2/repository/";
var langchain4jVersion = "1.9.1";
var langchain4jbeta = "1.9.1-beta17";
var langgraph4jVersion = "1.8.22";


Remove installed package from Jupyter cache


In [ ]:
%%bash 
rm -rf \{userHomeDir}/Library/Jupyter/kernels/rapaio-jupyter-kernel/mima_cache/org/bsc/langgraph4j


Add local Maven repo and resolve dependencies


In [ ]:
%dependency /add-repo local \{localRespoUrl} release|never snapshot|always
// %dependency /list-repos
%dependency /add org.slf4j:slf4j-jdk14:2.0.9
%dependency /add org.bsc.langgraph4j:langgraph4j-core:\{langgraph4jVersion}
%dependency /add org.bsc.langgraph4j:langgraph4j-langchain4j:\{langgraph4jVersion}
%dependency /add dev.langchain4j:langchain4j:\{langchain4jVersion}
%dependency /add dev.langchain4j:langchain4j-open-ai:\{langchain4jVersion}
%dependency /add net.sourceforge.plantuml:plantuml-mit:1.2024.8

%dependency /resolve


**Initialize Logger**


In [ ]:
try( var file = new java.io.FileInputStream("./logging.properties")) {
    java.util.logging.LogManager.getLogManager().readConfiguration( file );
}

var log = org.slf4j.LoggerFactory.getLogger("PlanAndExecute");


**Utility to render graph representation in PlantUML**


In [ ]:
import net.sourceforge.plantuml.SourceStringReader;
import net.sourceforge.plantuml.FileFormatOption;
import net.sourceforge.plantuml.FileFormat;

java.awt.Image plantUML2PNG( String code ) throws IOException { 
    var reader = new SourceStringReader(code);

    try(var imageOutStream = new java.io.ByteArrayOutputStream()) {

        var description = reader.outputImage( imageOutStream, 0, new FileFormatOption(FileFormat.PNG));

        var imageInStream = new java.io.ByteArrayInputStream(  imageOutStream.toByteArray() );

        return javax.imageio.ImageIO.read( imageInStream );

    }
}


## 1. Define the State

* `input` — user objective  
* `plan` — remaining steps (replaced on each planner/replan update)  
* `past_steps` — completed `(step, result)` pairs (appended)  
* `response` — final answer when the loop ends


In [ ]:
import org.bsc.langgraph4j.state.AgentState;
import org.bsc.langgraph4j.state.Channel;
import org.bsc.langgraph4j.state.Channels;

import java.util.ArrayList;
import java.util.List;
import java.util.Map;
import java.util.Optional;

record PastStep(String step, String result) implements java.io.Serializable {}

class PlanExecuteState extends AgentState {

    public static final String INPUT = "input";
    public static final String PLAN = "plan";
    public static final String PAST_STEPS = "past_steps";
    public static final String RESPONSE = "response";

    public static final Map<String, Channel<?>> SCHEMA = Map.of(
            INPUT, Channels.base(() -> ""),
            PLAN, Channels.base(ArrayList::new),
            PAST_STEPS, Channels.appender(ArrayList::new),
            RESPONSE, Channels.base(() -> "")
    );

    public PlanExecuteState(Map<String, Object> initData) {
        super(initData);
    }

    public String input() {
        return this.<String>value(INPUT).orElse("");
    }

    @SuppressWarnings("unchecked")
    public List<String> plan() {
        return this.<List<String>>value(PLAN).orElse(List.of());
    }

    @SuppressWarnings("unchecked")
    public List<PastStep> pastSteps() {
        return this.<List<PastStep>>value(PAST_STEPS).orElse(List.of());
    }

    public Optional<String> response() {
        return this.value(RESPONSE);
    }

    public boolean hasResponse() {
        return response().filter(r -> r != null && !r.isBlank()).isPresent();
    }
}


## 2. Stub tools

A tiny in-memory “search” tool keeps the notebook runnable offline.
Swap this for Tavily / HTTP / MCP tools when wiring a real LLM executor.


In [ ]:
import java.util.Locale;

class StubSearchTool {

    String search(String query) {
        var q = query == null ? "" : query.toLowerCase(Locale.ROOT);
        if (q.contains("weather") || q.contains("sf") || q.contains("san francisco")) {
            return "San Francisco: 60F and foggy.";
        }
        if (q.contains("nyc") || q.contains("new york")) {
            return "New York: 55F and cloudy.";
        }
        return "No structured result for: " + query;
    }
}

var searchTool = new StubSearchTool();


## 3. Define the Nodes (stub implementations)

> **TODO for live LLM:** replace each stub with LangChain4j `AiServices` (structured plan / replan)  
> and an agent executor that calls real tools. Keep the same state keys so the graph wiring stays unchanged.


In [ ]:
import org.bsc.langgraph4j.action.NodeAction;
import org.bsc.langgraph4j.action.EdgeAction;

import java.util.ArrayList;
import java.util.List;
import java.util.Map;

/**
 * Stub planner: produces a fixed two-step plan from the user input.
 * Live version should ask an LLM to emit ordered steps.
 */
class PlannerNode implements NodeAction<PlanExecuteState> {
    @Override
    public Map<String, Object> apply(PlanExecuteState state) {
        var goal = state.input();
        log.info("planner input: {}", goal);

        List<String> plan = List.of(
                "Gather facts relevant to: " + goal,
                "Synthesize a final answer for: " + goal
        );
        return Map.of(PlanExecuteState.PLAN, new ArrayList<>(plan));
    }
}

/**
 * Stub agent/executor: runs the first remaining plan step via StubSearchTool.
 */
class AgentNode implements NodeAction<PlanExecuteState> {
    private final StubSearchTool tool;

    AgentNode(StubSearchTool tool) {
        this.tool = tool;
    }

    @Override
    public Map<String, Object> apply(PlanExecuteState state) {
        var plan = state.plan();
        if (plan.isEmpty()) {
            return Map.of();
        }
        var step = plan.get(0);
        log.info("agent executing step: {}", step);

        var result = tool.search(step);
        return Map.of(
                PlanExecuteState.PAST_STEPS, new PastStep(step, result)
        );
    }
}

/**
 * Stub replanner:
 * - drops the step that was just executed
 * - when no steps remain, writes a final response from past_steps
 */
class ReplanNode implements NodeAction<PlanExecuteState> {
    @Override
    public Map<String, Object> apply(PlanExecuteState state) {
        var plan = new ArrayList<>(state.plan());
        var past = state.pastSteps();

        if (!plan.isEmpty()) {
            plan.remove(0);
        }

        if (plan.isEmpty()) {
            var sb = new StringBuilder("Final answer based on executed steps:\n");
            for (var ps : past) {
                sb.append("- ").append(ps.step()).append(" => ").append(ps.result()).append('\n');
            }
            var response = sb.toString().trim();
            log.info("replan -> respond");
            return Map.of(
                    PlanExecuteState.PLAN, plan,
                    PlanExecuteState.RESPONSE, response
            );
        }

        log.info("replan -> continue, remaining={}", plan);
        return Map.of(PlanExecuteState.PLAN, plan);
    }
}

EdgeAction<PlanExecuteState> shouldContinue = state ->
        state.hasResponse() ? "respond" : "continue";


## 4. Build and compile the graph


In [ ]:
import org.bsc.langgraph4j.StateGraph;
import org.bsc.langgraph4j.GraphRepresentation;
import static org.bsc.langgraph4j.action.AsyncNodeAction.node_async;
import static org.bsc.langgraph4j.action.AsyncEdgeAction.edge_async;
import static org.bsc.langgraph4j.StateGraph.START;
import static org.bsc.langgraph4j.StateGraph.END;

var planner = new PlannerNode();
var agent = new AgentNode(searchTool);
var replan = new ReplanNode();

var workflow = new StateGraph<>(PlanExecuteState.SCHEMA, PlanExecuteState::new)
        .addNode("planner", node_async(planner))
        .addNode("agent", node_async(agent))
        .addNode("replan", node_async(replan))
        .addEdge(START, "planner")
        .addEdge("planner", "agent")
        .addEdge("agent", "replan")
        .addConditionalEdges("replan", edge_async(shouldContinue), Map.of(
                "continue", "agent",
                "respond", END
        ));

var app = workflow.compile();


## 5. Visualize the graph


In [ ]:
var representation = workflow.getGraph( GraphRepresentation.Type.PLANTUML, "plan-and-execute", false );
display( plantUML2PNG( representation.getContent() ) );


## 6. Run (stub demo)

No API key required — the stub planner/agent/replan drive a full plan → execute → replan loop.


In [ ]:
var input = Map.<String,Object>of(
        PlanExecuteState.INPUT, "What is the weather in San Francisco?"
);

for (var event : app.stream(input)) {
    log.info("STEP: {}", event);
}


## 7. Next steps (live LLM)

1. Add `OpenAiChatModel` / Ollama (see `agentexecutor.ipynb`).
2. Replace `PlannerNode` / `ReplanNode` with structured LLM outputs (`List<String> plan` or `{plan|response}`).
3. Replace `AgentNode` with an agent-executor subgraph or tool-calling loop (`langgraph4j-agent-executor`).
4. Optionally add `MemorySaver` + `threadId` for resume / time-travel (see `persistence.ipynb`).

Keep contributing under [#8](https://github.com/langgraph4j/langgraph4j/issues/8) — Self-RAG, Reflection, and other tutorial ports follow the same how-to pattern.
